# Transcribe Audio with OpenAI Whisper

Upload an audio file (mp3, wav, m4a ...) and turn speech into text with the open-source Whisper model running inside Colab. It works for many languages, including Bengali; the transcript is saved as a text file with timestamps.

## How to run
- Open in Google Colab. Runtime type: **T4 GPU**.
- Run the cells from top to bottom.

_Rewritten from my original Colab notebook: `audiio transcribe.ipynb`._

In [ ]:
!pip install -q openai-whisper
!apt-get -qq install -y ffmpeg

## 1. Upload an audio file

In [ ]:
from google.colab import files
uploaded = files.upload()
audio_path = next(iter(uploaded))
print("Audio file:", audio_path)

## 2. Load the model
Sizes: `tiny`, `base`, `small`, `medium`, `large-v3`, `turbo`. Bigger is more accurate but slower.

In [ ]:
import torch
import whisper

device = "cuda" if torch.cuda.is_available() else "cpu"
model = whisper.load_model("small", device=device)
print("Using", device)

## 3. Transcribe
Set `LANGUAGE` (for example `"bn"` for Bengali, `"en"` for English) or `None` to auto-detect.

In [ ]:
LANGUAGE = "bn"
result = model.transcribe(audio_path, language=LANGUAGE, fp16=(device == "cuda"))
print("Detected language:", result["language"])
print(result["text"])

## 4. Save with timestamps and download

In [ ]:
def fmt(seconds):
    m, s = divmod(int(seconds), 60)
    return f"{m:02d}:{s:02d}"

with open("transcript.txt", "w", encoding="utf-8") as f:
    for seg in result["segments"]:
        f.write(f"[{fmt(seg['start'])} - {fmt(seg['end'])}] {seg['text'].strip()}\n")

files.download("transcript.txt")